# Onechess : données Leela et 8 experts (TPU Colab v5e-1)

Environnement d'exécution > Modifier le type > **TPU v5e-1**, puis Exécution > **Tout exécuter**. Garder l'onglet ouvert (~2 h). À la fin, les deux réseaux se téléchargent seuls (`leela.nnue`, `mat8.nnue`) : les laisser dans Téléchargements.

In [ ]:
!python -c "import jax; print(jax.devices())"

In [ ]:
import io, tarfile, urllib.request, glob, os
tarfile.open(fileobj=io.BytesIO(urllib.request.urlopen('https://codeload.github.com/ovnice-dev/onechess-tools/tar.gz/main').read())).extractall('/content/code')
TOOLS = glob.glob('/content/code/*/tools')[0]
INIT = glob.glob('/content/code/*/reseau/archives/final_v1.0.pt')[0]
print(TOOLS, INIT)
!pip -q install optax chess
!pip -q install torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
# Donnees Stockfish (memes que le temoin du 14-11 : 63,0 cp) et validation commune
!python -u {TOOLS}/hf_sources.py --url https://huggingface.co/datasets/official-stockfish/master-binpacks/resolve/main/nodes5000pv2_UHO.binpack --prefixe n5 --dest /content/donnees --octets 2000000000 --garde 1.0 --fixe 240 --graine 11 --parts 2 2>&1 | tail -3
import numpy as np
v = sorted(glob.glob('/content/donnees/*_p00_*.bin'))[0]
os.rename(v, '/content/donnees/z_val.bin')
np.fromfile('/content/donnees/z_val.bin', dtype='uint8').reshape(-1, 32)[:2_000_000].tofile('/content/donnees/z_val.bin')

In [ ]:
# 1) 8 experts : greffe 4 -> 8 sur le final, memes donnees et epoques que le temoin (63,0 cp)
COMMUN = f"--hidden 1024 --out-buckets 4 --chunk-files 4 --epoch-files 16 --init {INIT} --lr 3e-4 --val 2000000 --ft-dense 1 --limite-s 5400"
!python -u {TOOLS}/train_jax.py --stream '/content/donnees/*.bin' --buckets mat8 {COMMUN} --epochs 10 --out /content/mat8.nnue 2>&1 | grep -v '^ *$' | tail -14
from google.colab import files
files.download('/content/mat8.nnue')

In [ ]:
# 2) Donnees Leela (test80, filtrees et minimisees) : affinage du final, juge = match au temps
!python -u {TOOLS}/hf_sources.py --url https://huggingface.co/datasets/official-stockfish/master-binpacks/resolve/main/test80-2022-08-aug-16tb7p.v6-dd.min.binpack --prefixe l80 --dest /content/leela --octets 2000000000 --garde 1.0 --graine 11 --parts 2 2>&1 | tail -3
!cp /content/donnees/z_val.bin /content/leela/z_val.bin
!python -u {TOOLS}/train_jax.py --stream '/content/leela/*.bin' --buckets mat4 {COMMUN} --epochs 20 --out /content/leela.nnue 2>&1 | grep -v '^ *$' | tail -24
files.download('/content/leela.nnue')